## Matrix Solver Reference Implementation

### Floating Point Test

In [1]:
from ieee754 import double, IEEE754

# 直接获取双精度表示
print(double(13.375))

# 使用类获取更多信息
x = 13.375
a = IEEE754(x,precision=1) # 默认双精度
print(str(a)) # 二进制表示
print(a.hex()) # 十六进制表示
print(a.json()) # 详细信息(JSON)

0 10000000010 1010110000000000000000000000000000000000000000000000
0 10000010 10101100000000000000000
('41560000', ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'])
{'number': Decimal('13.375'), 'edge_case': False, 'sign_bit': '1', 'exponent_bits': 8, 'mantissa_bits': 23, 'total_bits': 32, 'sign': '0', 'scale': 3, 'scaled_number': 107, 'scaled_number_in_binary': '1101011', 'unable_to_scale': False, 'bias': 127, 'bias_in_binary': '01111111', 'exponent': '10000010', 'mantissa': '10101100000000000000000', 'mantissa_base_10': Decimal('0.67187500000000000000000'), 'result': '0 10000010 10101100000000000000000', 'hexadecimal': '41560000', 'hexadecimal_parts': ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'], 'converted_number': Decimal('13.375'), 'error': Decimal('0.000')}


In [2]:
a, b = 6.25,13.25
a_plus_b = a+b
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
rr = IEEE754(a_plus_b, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")

A        hex : 40C80000
B        hex : 41540000
Expected hex : 419C0000


In [3]:
a, b, c = 122.04, 0.06152, 0.237
r = a * b + c
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
cc = IEEE754(c, precision=1)
rr = IEEE754(r, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"C        hex : {cc.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")
print(r)
print(a/b)

A        hex : 42F4147B
B        hex : 3D7BFC65
C        hex : 3E72B021
Expected hex : 40F7D63B
7.7449008
1983.7451235370613


### LU Decomposition

In [4]:
import numpy as np

# available floating-point primitives

cycles = 0

verbose = 3


# cycle: 0
def neg(v: float) -> float:
    return -v


# cycle: 16
def fma(a: float, b: float, c: float) -> float:
    global cycles
    global verbose
    cycles += 16
    if verbose > 2:
        print(f"[fma] {a:.3f} * {b:.3f} + {c:.3f} = {a * b + c:.3f}")
    return a * b + c


# cycle: 28
def div(a: float, b: float) -> float:
    global cycles
    global verbose
    cycles += 28
    if verbose > 2:
        print(f"[div] {a:.3f} / {b:.3f} = {a / b:.3f}")
    return a / b


def lu_decomp(A: np.ndarray):
    n = A.shape[0]
    # L = np.eye(n, dtype=float)
    # U = np.zeros((n, n), dtype=float)
    L = np.random.rand(n, n)
    U = np.random.rand(n, n)
    for j in range(n):
        for i in range(n):
            if i <= j:
                U[i, j] = neg(A[i, j])
                for k in range(i):
                    # fused multiply and add
                    U[i, j] = fma(L[i, k], U[k, j], U[i, j])  # k < i
                U[i, j] = neg(U[i, j])
            else:  # i > j
                L[i, j] = neg(A[i, j])
                for k in range(j):
                    L[i, j] = fma(L[i, k], U[k, j], L[i, j])  # k < j; i > j => k < i
                L[i, j] = div(L[i, j], U[j, j])
                L[i, j] = neg(L[i, j])
    return L, U


def lu_decomp_more_hard(A: np.ndarray):
    n = A.shape[0]
    LU = np.random.rand(n, n)
    global cycles
    global verbose
    cycles = 0

    def fetch_A(i, j):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"fetch  A i={i} j={j} => {A[i, j]:.3f}")
        return A[i, j]

    def fetch_LU(i, j):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"fetch LU i={i} j={j} => {LU[i, j]:.3f}")
        return LU[i, j]

    def store_LU(i, j, v):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"store LU i={i} j={j} => {v:.3f}")
        LU[i, j] = v


    for idx_j in range(n):
        for idx_i in range(n):
            # module : start busy done
            f32_1 = fetch_A(idx_i, idx_j)  # 2 cycles {StateFetchA + StateWaitA}
            f32_1 = -f32_1                              # combinational [f32_1_neg] {}
            idx_m = idx_j if idx_i > idx_j else idx_i   # combinational [min_i_j]
            for idx_k in range(idx_m):
                f32_2 = fetch_LU(i=idx_i, j=idx_k)  # 2 cycles {StateFetchL + StateWaitL}
                f32_3 = fetch_LU(i=idx_k, j=idx_j)  # 2 cycles {StateFetchU + StateWaitU}
                f32_1 = fma(f32_2, f32_3, f32_1)  # 16 cycles {StateStartFMA + StateWaitFMA}
            if idx_i > idx_j:
                f32_2 = fetch_LU(i=idx_j, j=idx_j)  # 2 cycles {StateFetchDiag + StateWaitDiag}
                f32_1 = div(a=f32_1, b=f32_2)  # 28 cycles {StateStartDiv + StateWaitDiv}
            f32_1 = -f32_1                              # combinational [f32_1_neg]
            store_LU(i=idx_i, j=idx_j, v=f32_1)  # 2 cycles {StateStore + StateWaitStore}

    print(f"cycle = {cycles}")

    return LU

StateIdle
StateFetchA
StateWaitA
StateInitK
StateFetchL
StateWaitL
StateFetchU
StateWaitU
StateStartFMA
StateWaitFMA
StateCheckK
StateFetchDiag
StateWaitDiag
StateStartDiv
StateWaitDiv
StateStore
StateWaitStore
StateAdvance
StateDone

In [5]:
cycles = 0; mat = np.random.rand(64, 64); lu_decomp(mat); print(f"64x64 cycles={cycles}")
cycles = 0; mat = np.random.rand(32, 32); lu_decomp(mat); print(f"32x32 cycles={cycles}")
cycles = 0; mat = np.random.rand(16, 16); lu_decomp(mat); print(f"16x16 cycles={cycles}")
cycles = 0; mat = np.random.rand(8, 8); lu_decomp(mat); print(f"8x8 cycles={cycles}")

[div] -0.317 / 0.742 = -0.427
[div] -0.461 / 0.742 = -0.620
[div] -0.826 / 0.742 = -1.112
[div] -0.785 / 0.742 = -1.057
[div] -0.595 / 0.742 = -0.801
[div] -0.195 / 0.742 = -0.263
[div] -0.928 / 0.742 = -1.250
[div] -0.530 / 0.742 = -0.714
[div] -0.826 / 0.742 = -1.112
[div] -0.934 / 0.742 = -1.258
[div] -0.796 / 0.742 = -1.073
[div] -0.108 / 0.742 = -0.145
[div] -0.396 / 0.742 = -0.533
[div] -0.128 / 0.742 = -0.173
[div] -0.166 / 0.742 = -0.224
[div] -0.660 / 0.742 = -0.889
[div] -0.829 / 0.742 = -1.117
[div] -0.958 / 0.742 = -1.291
[div] -0.049 / 0.742 = -0.067
[div] -0.063 / 0.742 = -0.085
[div] -0.987 / 0.742 = -1.329
[div] -0.119 / 0.742 = -0.160
[div] -0.452 / 0.742 = -0.609
[div] -0.786 / 0.742 = -1.059
[div] -0.157 / 0.742 = -0.212
[div] -0.182 / 0.742 = -0.245
[div] -0.042 / 0.742 = -0.056
[div] -0.557 / 0.742 = -0.750
[div] -0.514 / 0.742 = -0.692
[div] -0.425 / 0.742 = -0.572
[div] -0.814 / 0.742 = -1.097
[div] -0.477 / 0.742 = -0.643
[div] -0.224 / 0.742 = -0.302
[div] -0.9

In [6]:
import scripts.matrix2mem

mat = np.random.rand(8, 8)
# mat = np.array(
#     [
#         [4.0, 3.0, 2.0, 1.0],
#         [3.0, 5.0, 1.5, -2.0],
#         [2.0, 1.5, 6.0, 0.5],
#         [1.0, -2.0, 0.5, 7.0],
#     ],
#     dtype=np.float32,
# )
print(f"mat:{mat}")
scripts.matrix2mem.matrix_to_word_mem(mat, "tmp/lu_test_input_8x8.mem")

mat:[[0.38073899 0.229543   0.0341203  0.00834842 0.86124911 0.06773356
  0.82497696 0.78304735]
 [0.61352497 0.78680377 0.31636217 0.18088008 0.16707004 0.71065159
  0.35783092 0.65646397]
 [0.19059683 0.1994136  0.08395937 0.60416145 0.45849594 0.12988391
  0.62566174 0.93887163]
 [0.9668346  0.08016566 0.2097322  0.12684923 0.10297271 0.50003713
  0.45964055 0.22206603]
 [0.870573   0.56265115 0.8089117  0.10512068 0.46154294 0.06981657
  0.99528496 0.97273764]
 [0.13105295 0.96228256 0.35459845 0.638028   0.316877   0.40202278
  0.16652084 0.44447763]
 [0.74747312 0.49573126 0.9340839  0.87487663 0.18471664 0.26336293
  0.56843645 0.53572098]
 [0.72935292 0.09399555 0.11617672 0.68995758 0.27330198 0.10217628
  0.22549836 0.25856681]]
Wrote 64 words to tmp\lu_test_input_8x8.mem
Matrix shape: (8, 8)
Flatten order: row-major


In [7]:
lu = lu_decomp_more_hard(mat)
uu = np.triu(lu)
ll = np.tril(lu, k=-1) + np.eye(lu.shape[0])
print(f"L:{ll}\nU:{uu}\nLU:{lu}")
print(f"L@U:{ll @ uu}")
scripts.matrix2mem.matrix_to_word_mem(lu, "tmp/lu_test_expected_8x8.mem")

fetch  A i=0 j=0 => 0.381
store LU i=0 j=0 => 0.381
fetch  A i=1 j=0 => 0.614
fetch LU i=0 j=0 => 0.381
[div] -0.614 / 0.381 = -1.611
store LU i=1 j=0 => 1.611
fetch  A i=2 j=0 => 0.191
fetch LU i=0 j=0 => 0.381
[div] -0.191 / 0.381 = -0.501
store LU i=2 j=0 => 0.501
fetch  A i=3 j=0 => 0.967
fetch LU i=0 j=0 => 0.381
[div] -0.967 / 0.381 = -2.539
store LU i=3 j=0 => 2.539
fetch  A i=4 j=0 => 0.871
fetch LU i=0 j=0 => 0.381
[div] -0.871 / 0.381 = -2.287
store LU i=4 j=0 => 2.287
fetch  A i=5 j=0 => 0.131
fetch LU i=0 j=0 => 0.381
[div] -0.131 / 0.381 = -0.344
store LU i=5 j=0 => 0.344
fetch  A i=6 j=0 => 0.747
fetch LU i=0 j=0 => 0.381
[div] -0.747 / 0.381 = -1.963
store LU i=6 j=0 => 1.963
fetch  A i=7 j=0 => 0.729
fetch LU i=0 j=0 => 0.381
[div] -0.729 / 0.381 = -1.916
store LU i=7 j=0 => 1.916
fetch  A i=0 j=1 => 0.230
store LU i=0 j=1 => 0.230
fetch  A i=1 j=1 => 0.787
fetch LU i=1 j=0 => 1.611
fetch LU i=0 j=1 => 0.230
[fma] 1.611 * 0.230 + -0.787 = -0.417
store LU i=1 j=1 => 0.41

In [8]:
import ast

source = """
def lu_kernel(n: i32):
    for j in range(n):
        for i in range(n):
            acc = -fetch_A(i, j)
            if i > j:
                acc = div(acc, fetch_LU(j, j))
            store_LU(i, j, acc)
"""

tree = ast.parse(source)
print(ast.dump(tree, indent=2))


Module(
  body=[
    FunctionDef(
      name='lu_kernel',
      args=arguments(
        args=[
          arg(
            arg='n',
            annotation=Name(id='i32', ctx=Load()))]),
      body=[
        For(
          target=Name(id='j', ctx=Store()),
          iter=Call(
            func=Name(id='range', ctx=Load()),
            args=[
              Name(id='n', ctx=Load())]),
          body=[
            For(
              target=Name(id='i', ctx=Store()),
              iter=Call(
                func=Name(id='range', ctx=Load()),
                args=[
                  Name(id='n', ctx=Load())]),
              body=[
                Assign(
                  targets=[
                    Name(id='acc', ctx=Store())],
                  value=UnaryOp(
                    op=USub(),
                    operand=Call(
                      func=Name(id='fetch_A', ctx=Load()),
                      args=[
                        Name(id='i', ctx=Load()),
                        Name(i

In [9]:
grid_width = 8
grid_height = 8


ports = [
    [
        [
            False,  # down (1,0)
            False,  # right (0,1)
            False,  # up (-1,0)
            False,  # left (0,-1)
        ]
        for j in range(grid_height)
    ]
    for i in range(grid_width)
]

dir_nxt = ((0, 1), (1, 0), (0, -1), (-1, 0))
opp_dir = (2, 3, 0, 1)

netlist_test_input_path = "data/netlist_test_data.txt"


def bitstr2bittuple(s):
    return tuple((s[i] == "1" for i in range(4)))


with open(netlist_test_input_path, "r") as input_file:
    line_n = 0
    while True:
        line = input_file.readline()
        if not line:
            break
        line_split = line.strip().split(" ")
        i = 0
        for cell in line_split:
            ports[i][line_n] = bitstr2bittuple(cell)
            i += 1
        line_n += 1
# print(ports)


def getwireports(i, j):
    return ports[i][j]


def iswire(i, j):
    return ports[i][j] != (0, 0, 0, 0)

result = [["N" for j in range(grid_height)] for i in range(grid_width)]
def bfs():
    # visited : reg
    visited = [[False for j in range(grid_height)] for i in range(grid_width)]
    queue = []
    node_idx = 0

    def add_to_queue(i, j):
        # print(f"add {(i, j)} to queue")
        visited[i][j] = True
        result[i][j] = node_idx
        for dir in range(4):
            if ports[i][j][dir]:
                queue.insert(0, (i, j, dir))

    for j in range(grid_height):
        for i in range(grid_width):
            if not visited[i][j] and iswire(i, j):
                node_idx += 1
                add_to_queue(i, j)
                # print(f"begin exploring from {(i, j)}")
                while len(queue) > 0:
                    ci, cj, dir = queue.pop()
                    dnxt = dir_nxt[dir]
                    ni, nj = ci + dnxt[0], cj + dnxt[1]
                    # print(f"at {(ci, cj)} dir {dir} => check {(ni, nj)}")
                    if (
                        not visited[ni][nj]
                        and iswire(i, j)
                        and ni >= 0
                        and nj >= 0
                        and ni < grid_width
                        and nj < grid_height
                    ):
                        if getwireports(ni, nj)[opp_dir[dir]]:
                            add_to_queue(ni, nj)
                    # if checknxt(*cur_explore):
def print_node_map():
    for j in range(grid_height):
        for i in range(grid_width):
            print(f"{result[i][j]} ",end='')
        print("")
bfs()
print_node_map()

IndexError: string index out of range

In [ ]:
def bfs_hard_1():
    reset()
    node_idx = 0
    for j in range(grid_height):
        for i in range(grid_width):
            if check_1(i,j): # visited implemented as ram, wire have to check port ram (2 cycle)
                node_idx += 1 # combinational
                add_to_queue(i, j) # queue implemented as ram + reg idx (not optimized -- 4 cycles)
                while queue_len > 0:
                    target_dir,ni,nj,real_dir = pop_queue() # 4 cycles
                    if check_2(target_dir,ni,nj,real_dir): # visited / wire stuff (2 cycles)
                        add_to_queue(ni, nj) # 4 cycles

CellVisStore


ComponentStore
  - ports (from RAM 1 & RAM 2) 4b per cell

- Substore for netlist generation
  - visited (internal) 1b per cell
  - queue

In [11]:
import struct


def ascii_hex_to_float32(s: str, byteorder: str = "big") -> float:
    """
    将表示 32-bit IEEE754 浮点数的 8 位十六进制 ASCII 字符串
    转成 Python float。

    例如:
        "3f800000" -> 1.0
    """
    if len(s) != 8:
        raise ValueError("输入必须是 8 个 ASCII 字符")

    raw = bytes.fromhex(s)

    if byteorder == "big":
        return struct.unpack(">f", raw)[0]
    elif byteorder == "little":
        return struct.unpack("<f", raw)[0]
    else:
        raise ValueError("byteorder 只能是 'big' 或 'little'")


In [13]:
ascii_hex_to_float32("40555555")

3.3333332538604736

In [ ]:
a = """
11111000000000000000000000000000
11111100000000000000000000000000
00011111000000000000000000000000
00001111100000000000000000000000
00000011111000000000000000000000
00000001111100000000000000000000
00000000111100000000000000000000
00000000001110000000000000000000
00000000001111000000000000000000
11000000000111000000000000000000
11100000000111100000000000000000
11110000000011100000000000000000
11111000000011100000000000000000
01111100000011111111111111111111
11111110000011111111111111111111
11111111000011111111111111111111
11111111000011111111111111111111
11111110000011111111111111111111
01111100000011111111111111111111
11111000000011100000000000000000
11110000000011100000000000000000
11100000000111100000000000000000
11000000000111100000000000000000
00000000001111000000000000000000
00000000001110000000000000000000
00000000111110000000000000000000
00000001111100000000000000000000
00000011111000000000000000000000
00001111100000000000000000000000
00011111000000000000000000000000
11111110000000000000000000000000
11111000000000000000000000000000
"""

for row in range(32):
  for col in range(32):
    print(a[row*33+32-col],end='')
  print("\n",end='')

00000000000000000000000000011111
00000000000000000000000000111111
00000000000000000000000011111000
00000000000000000000000111110000
00000000000000000000011111000000
00000000000000000000111110000000
00000000000000000000111100000000
00000000000000000001110000000000
00000000000000000011110000000000
00000000000000000011100000000011
00000000000000000111100000000111
00000000000000000111000000001111
00000000000000000111000000011111
11111111111111111111000000111110
11111111111111111111000001111111
11111111111111111111000011111111
11111111111111111111000011111111
11111111111111111111000001111111
11111111111111111111000000111110
00000000000000000111000000011111
00000000000000000111000000001111
00000000000000000111100000000111
00000000000000000111100000000011
00000000000000000011110000000000
00000000000000000001110000000000
00000000000000000001111100000000
00000000000000000000111110000000
00000000000000000000011111000000
00000000000000000000000111110000
00000000000000000000000011111000
0000000000